<a href="https://colab.research.google.com/github/janepium/deteccion-anomalias-rt-iot/blob/main/notebooks/08_comparacion_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importar librerías


In [7]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

#Cargar el dataset

In [8]:
df = pd.read_csv(
    "dataset_limpio.csv",
    index_col=0
)

print("Dimensiones:", df.shape)
df.head()

Dimensiones: (117922, 37)


,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,flow_pkts_per_sec,down_up_ratio,...,fwd_iat.std,fwd_subflow_bytes,fwd_bulk_bytes,active.std,idle.tot,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,38667,1883,tcp,mqtt,32.011598,9,5,3,0.437341,0.555556,...,1.040307e+07,25.333333,0.0,0.0,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,51143,1883,tcp,mqtt,31.883584,9,5,3,0.439097,0.555556,...,1.046346e+07,25.333333,0.0,0.0,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,44761,1883,tcp,mqtt,32.124053,9,5,3,0.435811,0.555556,...,1.044238e+07,24.666667,0.0,0.0,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,60893,1883,tcp,mqtt,31.961063,9,5,3,0.438033,0.555556,...,1.048253e+07,24.666667,0.0,0.0,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,51087,1883,tcp,mqtt,31.902362,9,5,3,0.438839,0.555556,...,1.044702e+07,25.333333,0.0,0.0,2.981470e+07,0.0,64240,26847,502,MQTT_Publish


#Separar X e y

Primero excluyes las variables leaky.

In [9]:
leaky = [
    "id.orig_p",
    "id.resp_p",
    "proto",
    "service"
]

In [10]:
X = df.drop(columns=["Attack_type"])
y = df["Attack_type"]

X = X.drop(columns=leaky)

print("X:", X.shape)
print("y:", y.shape)

X: (117922, 32)
y: (117922,)


#Train/Test

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Decision Tree

In [12]:
dt = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

# Random Forest

In [13]:
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Logistic Regression

In [14]:
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

# KNN

In [15]:
knn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(
        n_neighbors=5
    ))
])

# XGBoost

In [16]:
try:
    from xgboost import XGBClassifier
    xgb_disponible = True
    print("XGBoost disponible")
except ImportError:
    xgb_disponible = False
    print("XGBoost no está disponible")

XGBoost disponible


# Función para evaluar

In [17]:
def evaluar_modelo(nombre, modelo):

    inicio = time.perf_counter()

    modelo.fit(X_train, y_train)

    tiempo_entrenamiento = time.perf_counter() - inicio

    inicio = time.perf_counter()

    y_pred = modelo.predict(X_test)

    tiempo_prediccion = time.perf_counter() - inicio

    resultado = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "F1-score": f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "F1-Macro": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Tiempo entrenamiento (s)": tiempo_entrenamiento,
        "Tiempo predicción (s)": tiempo_prediccion
    }

    return resultado

# Ejecutar los modelos

In [22]:
modelos = {
    "Decision Tree": dt,
    "Random Forest": rf,
    "Logistic Regression": lr,
    "KNN": knn
}

In [19]:
resultados = []

for nombre, modelo in modelos.items():

    print(f"Entrenando {nombre}...")

    resultado = evaluar_modelo(
        nombre,
        modelo
    )

    resultados.append(resultado)

Entrenando Decision Tree...
Entrenando Random Forest...
Entrenando Logistic Regression...
Entrenando KNN...


# Crear la tabla comparativa

In [20]:
resultados_df = pd.DataFrame(resultados)

resultados_df

,Modelo,Accuracy,Precision,Recall,F1-score,F1-Macro,Tiempo entrenamiento (s),Tiempo predicción (s)
0,Decision Tree,0.995718,0.962499,0.982527,0.995743,0.971244,0.957337,0.008149
1,Random Forest,0.996184,0.964506,0.987651,0.996212,0.974786,5.233917,0.132173
2,Logistic Regression,0.957346,0.709240,0.931045,0.957574,0.744948,83.531605,0.021135
3,KNN,0.994149,0.953223,0.973892,0.994165,0.962462,0.187190,17.569237
